## Домашнее задание 2. Tiny ImageNet Challenge (10 баллов + 1 бонус)

В этом задании Вам предстоит обучить свёрточную нейросеть для решения задачи мультиклассовой классификации на датасете [Tiny ImageNet](https://www.kaggle.com/c/tiny-imagenet) (200 классов, по 500 изображений на класс в трейне и по 50 в валидации и тесте).

``Ссылка на наше соревнование:`` [тык](https://www.kaggle.com/t/b801ab030059413e8961b10dd86b4822).

## Критерии оценки

* Без отчёта с графиками лосса и метрики (``accuracy@1``) на обучении работа **не принимается!**
* Используйте **интерактивные** (не изобретайте велосипед с помощью `plt.plot`) инструменты для просмотра прогресса, например, TensorBoard или Wandb.   
    *В Wandb также можно писать отчёты по вашим данным, попробуйте, это очень экономит время.*

* Баллы выставляются на основе Private leaderboard
    - $\geq$ 0.45 - 10 баллов,
    - $\geq$ 0.35 - 6 баллов,
    - $\geq$ 0.25 - 3 балла.

* За лучший результат на Private leaderboard +1 балл

## Объяснение оценок

* *Тест*: это часть набора данных, идентичная валидации, но лейблы известны только нам.
* *Как отправить*:
   * Не меняйте этот ноутбук, ваш код должен отработать в нём для корректной проверки инференса. Обучать можно как угодно, например, в нём же с флагом `DO_TRAIN=True`, в своём ноутбуке или из консоли.
   * После того, как вы обучили свою сеть, [сохраните веса](https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html) в «*checkpoint.pth*» с помощью `model.state_dict()` и ` torch.save()`.
   * Установите `DO_TRAIN = False`, нажмите «Перезапустить и запустить все ячейки» и убедитесь, что точность проверки на валидации рассчитана правильно. **Учитывайте, что вам нужен чекпоинт модели**.
   * Загрузите «*checkpoint.pth*» на Google Диск, скопируйте на него ссылку, доступную только для просмотра, и вставьте ее также в «*solution.py*».

* *Отчет*: PDF, свободная форма (можно написать в Markdown или .ipynb, главное сконвертировать в PDF в конце; отчёт в Wandb просто присылайте ссылкой), следует упомянуть:
   * Ваша история настроек и улучшений. Как вы начинали, что искали. (*Я проанализировал те и эти документы/источники/репорты/статьи. Я попробовал то и это, чтобы адаптировать их к моей задаче. ...*)
   * Какие архитектуры вы пробовали? Какие из них не сработали и почему, по вашему мнению? Какую выбрали на финальный сабмит и почему?
   * То же самое касается метода обучения (batch size, алгоритм оптимизации, количество итераций...): что и почему?
   * То же самое касается методов предотвращения переобучения (регуляризации). Какие из них вы пробовали? Каковы были их последствия и можете ли вы объяснить, почему?
   * **Самое главное**: вы получили глубокие знания. Можете ли вы отрефлексировать и привести несколько примеров того, как опыт этого упражнения повлияет на ваше обучение будущих нейронных сетей? (хитрости, эвристики, выводы, наблюдения)
   * **Перечислите и сошлитесь на все внешние источники кода, если вы их использовали**.
* *Инструмент логгирования*: дополните отчет скриншотами графиков точности и лосса (на трейне и на валидации) с течением времени.

## Можно:

* Писать свои модели
* Использовать готовые реализации архитектур
* Использовать дополнительные данные 

## Нельзя:

* Переиспользовать любые предобученные веса
* Увеличивать изображения (например, не изменять их размер до $224 \times 224$ или $256 \times 256$).
* Делиться сабмитами

## Советы

* **Одно изменение за раз**: не тестируйте несколько новых вещей одновременно (если вы не очень уверены, что они будут работать). Обучите модель, внесите одно изменение, обучите снова.
* Много гуглите: постарайтесь изобрести как можно меньше велосипедов. Черпайте вдохновение из туториалов PyTorch, GitHub, блогов...
* Используйте сверточные архитектуры.
* Используйте графический процессор.
* Регуляризация очень важна: L2, batch norm, early stopping, аугментации, семплирование...
* Уделяйте большое внимание графикам точности и потерь (например, в wandb). Отслеживайте неудачи как можно раньше, прекращайте неудачные эксперименты как можно раньше.
* 2-3 часов обучения (в Colab) должно быть достаточно для большинства моделей, возможно, 4-6 часов, если вы экспериментируете.
* Время от времени сохраняйте чекпоинты вместе со стейтом оптимизатора на случай, если что-то пойдет не так (оптимизация расходится, Colab отключается...).
* Не используйте слишком большие батчи, они могут работать медленно и требовать много памяти. Это справедливо и для инференса.
* Также не забудьте использовать `torch.no_grad()` и `.eval()` во время инференса.

In [1]:
import random
from tqdm import tqdm
import numpy as np
import pandas as pd
import wandb

import torch
from torch import nn
import torch.optim as optim
from torchvision import transforms

from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

In [2]:
def set_global_seed(seed: int) -> None:
    """Set global seed for reproducibility.
    :param int seed: Seed to be set
    """
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_global_seed(42)

In [3]:
# If `True`, will train the model from scratch and validate it.
# If `False`, instead of training will load weights from './checkpoint.pth'.
# When grading, we will test both cases.
DO_TRAIN = False

root_datasets = "./"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
def get_dataloader(path, kind):
    """
    Return dataloader for a `kind` split of Tiny ImageNet.
    If `kind` is 'val' or 'test', the dataloader should be deterministic.
    path:
        `str`
        Path to the dataset root - a directory which contains 'train' and 'val' folders.
    kind:
        `str`
        'train', 'val' or 'test'

    return:
    dataloader:
        `torch.utils.data.DataLoader` or an object with equivalent interface
        For each batch, should yield a tuple `(preprocessed_images, labels)` where
        `preprocessed_images` is a proper input for `predict()` and `labels` is a
        `torch.int64` tensor of shape `(batch_size,)` with ground truth class labels.
    """
    # Your code here
    IMAGE_NET_MEAN = np.array([0.485, 0.456, 0.406])
    IMAGE_NET_STD  = np.array([0.229, 0.224, 0.225])

    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_NET_MEAN, IMAGE_NET_STD),
    ])


    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_NET_MEAN, IMAGE_NET_STD)
    ])
    
    if kind == 'train':
        dataset    = ImageFolder(f"{path}/train/", transform=train_transform)
        dataloader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)
        
        return dataloader
    
    
    dataset    = ImageFolder(f"{path}/{kind}/", transform=val_transform)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False, drop_last=False)
    
    return dataloader

def get_model():
    """
    Create neural net object, initialize it with raw weights, upload it to GPU.

    return:
    model:
        `torch.nn.Module`
    """
    # Your code here
    
    model = nn.Sequential(
        nn.Flatten(start_dim=1),
        nn.Linear(64 * 64 * 3, 128), nn.BatchNorm1d(128), nn.ReLU(),
        nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(),
        nn.Linear(128, 200),
    )
    
    return model

def get_optimizer(model):
    """
    Create an optimizer object for `model`, tuned for `train_on_tinyimagenet()`.

    return:
    optimizer:
        `torch.optim.Optimizer`
    """
    # Your code here
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)
    
    return optimizer

def load_weights(model, checkpoint_path):
    """
    Initialize `model`'s weights from `checkpoint_path` file.

    model:
        `torch.nn.Module`
        See `get_model()`.
    checkpoint_path:
        `str`
        Path to the checkpoint.
    """
    # Your code here
    
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    
    model.load_state_dict(checkpoint['model'])
    model.eval()


In [5]:
# Initialize dataloaders
train_dataloader = get_dataloader(f"{root_datasets}/tiny-imagenet-200/", 'train')
val_dataloader   = get_dataloader(f"{root_datasets}/tiny-imagenet-200/", 'val')
test_dataloader  = get_dataloader(f"{root_datasets}/tiny-imagenet-200/", 'test')

# Initialize the raw model
model = get_model()

In [6]:
@torch.no_grad()
def predict(model, batch):
    """
    model:
        `torch.nn.Module`
        The neural net, as defined by `get_model()`.
    batch:
        unspecified
        A batch of Tiny ImageNet images, as yielded by `get_dataloader(..., 'val')`
        (with same preprocessing and device).

    return:
    prediction:
        `torch.tensor`, shape == (N, 200), dtype == `torch.float32`
        The scores of each input image to belong to each of the dataset classes.
        Namely, `prediction[i, j]` is the score of `i`-th minibatch sample to
        belong to `j`-th class.
        These scores can be 0..1 probabilities, but for better numerical stability
        they can also be raw class scores after the last (usually linear) layer,
        i.e. BEFORE softmax.
    """
    X = batch[0].to(device)
    out = model(X)
    return torch.argmax(out, 1)


@torch.no_grad()
def validate(dataloader, model):
    """
    Run `model` through all samples in `dataloader`, compute accuracy and loss.

    dataloader:
        `torch.utils.data.DataLoader` or an object with equivalent interface
        See `get_dataloader()`.
    model:
        `torch.nn.Module`
        See `get_model()`.

    return:
    accuracy:
        `float`
        The fraction of samples from `dataloader` correctly classified by `model`
        (top-1 accuracy). `0.0 <= accuracy <= 1.0`
    loss:
        `float`
        Average loss over all `dataloader` samples.
    """
    # Your code here
    model.eval()
    accuracy, loss, count = 0, 0, 0
    loss_fn = torch.nn.CrossEntropyLoss(reduction="sum")
    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)
        
        out = model(X)
        
        loss     += loss_fn(out, y).item()
        accuracy += torch.sum(torch.argmax(out, 1) == y).item()
        count    += X.shape[0]

    return loss / count, accuracy / count

def train_on_tinyimagenet(train_dataloader, val_dataloader, model, optimizer):
    """
    Train `model` on `train_dataloader` using `optimizer`. Use best-accuracy settings.

    train_dataloader:
    val_dataloader:
        See `get_dataloader()`.
    model:
        See `get_model()`.
    optimizer:
        See `get_optimizer()`.
    """
   
    model.to(device)
    loss_fn = torch.nn.CrossEntropyLoss()
    
    wandb.init(project="TinyImageNet-baseline")
    
    global_step = 0
    best_acc = None
    for _ in tqdm(range(3)):
        model.train()
        
        for X, y in train_dataloader:
            global_step += 1
            
            optimizer.zero_grad()
            X = X.to(device)
            y = y.to(device)
            
            out  = model(X)
            loss = loss_fn(out, y)
            loss.backward()
            
            optimizer.step()
            
            acc = torch.sum(torch.argmax(out, 1) == y) / y.shape[0]
            wandb.log({"train/loss": loss.item(), "train/accuracy": acc.item()}, step=global_step)
        
        eval_loss, eval_acc = validate(dataloader=val_dataloader, model=model)
        wandb.log({"eval/loss": eval_loss, "eval/accuracy": eval_acc}, step=global_step)
        if best_acc is None or best_acc < eval_acc:
            torch.save({
                "model"    : model.state_dict(),
                "optimizer": optimizer.state_dict()
            }, "checkpoint.pth")
            
    wandb.finish()

In [7]:
if DO_TRAIN:
    # Train from scratch
    optimizer = get_optimizer(model)
    train_on_tinyimagenet(train_dataloader, val_dataloader, model, optimizer)
else:
    # Finally load weights
    load_weights(model, "./checkpoint.pth")

/tmp/ipykernel_61376/3283697659.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")


In [8]:
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=12288, out_features=128, bias=True)
  (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=128, bias=True)
  (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Linear(in_features=128, out_features=200, bias=True)
)

In [9]:
example_batch, example_batch_labels = next(iter(train_dataloader))

In [10]:
# Classify some validation samples
example_batch, example_batch_labels = next(iter(val_dataloader))
model.eval()
with torch.no_grad():
    example_predicted_labels = predict(model, [example_batch])

print("Predicted class / Ground truth class")
for predicted, gt in list(zip(example_predicted_labels, example_batch_labels))[:15]:
    print("{:03d} / {:03d}".format(predicted, gt))

Predicted class / Ground truth class
078 / 000
097 / 000
011 / 000
126 / 000
189 / 000
022 / 000
044 / 000
014 / 000
045 / 000
093 / 000
167 / 000
082 / 000
078 / 000
010 / 000
036 / 000


In [11]:
validate(val_dataloader, model)

(6.398388494873047, 0.0031)

In [12]:
# Print validation accuracy
val_loss, val_accuracy =  validate(val_dataloader, model)
val_accuracy *= 100

print("Validation accuracy: %.2f%%" % val_accuracy)

Validation accuracy: 0.31%


In [13]:
map_classes = {class_idx: class_name for class_name, class_idx in train_dataloader.dataset.class_to_idx.items()}

In [14]:
# example real submission
pred_dict = {}
pred_labels = []
model.eval()

for batch, _ in tqdm(test_dataloader):
    with torch.no_grad():
        predicted_labels = predict(model, [batch])
    pred_labels.extend(predicted_labels.tolist())
for i, (img_name, _) in enumerate(test_dataloader.dataset.imgs):
    pred_dict[img_name.split("/")[-1]] = map_classes[pred_labels[i]]

100%|███████████████████████████████████████████████████| 157/157 [00:03<00:00, 51.04it/s]


In [15]:
submission_df = pd.DataFrame(pred_dict.items(), columns=["id", "pred"])
submission_df.to_csv("baseline_submission.csv", index=False)